In [1]:
import pandas as pd
import os
import re
import numpy as np

frequency = 64
path = f"data/dreamt/data_{frequency}Hz"
cols_na = ["Obstructive_Apnea", "Central_Apnea", "Hypopnea", "Multiple_Events"]
nb_users_max = 20
X_all_patients = []
y_all_patients = []
prog = re.compile(r""".*S(\d\d\d).*""")
for name in os.listdir(path):
    if os.path.isfile(os.path.join(path, name)):
        id_user = int(prog.match(os.path.join(path, name)).group(1))
        if id_user < nb_users_max:
            # print(id_user)
            df = pd.read_csv(os.path.join(path, name))
            # df[cols_na] = df[cols_na].fillna(0.0)
            df["Sleep_Stage"] = df["Sleep_Stage"].replace("P", "W")
            df = df.drop(
                columns=[
                    "IBI",
                    "TIMESTAMP",
                    "Obstructive_Apnea",
                    "Central_Apnea",
                    "Hypopnea",
                    "Multiple_Events",
                ]
            )

            y_all_patients.append(df.Sleep_Stage.to_numpy())
            X_all_patients.append(df.drop(columns=["Sleep_Stage"]).to_numpy())


In [2]:
X = []
y = []

for X_p, y_p in zip(X_all_patients, y_all_patients):
    X.append(X_p[:-1].reshape(-1, 64, 7))  # remove the last point to get 64 windows
    y.append(y_p[:-1].reshape(-1, 64))

In [ ]:
from sklearn.model_selection import train_test_split

X_train = []
X_test = []
y_train = []
y_test = []

for x_i, y_i in zip(X, y):
    split = train_test_split(x_i, y_i, test_size=0.2, shuffle=True)
    X_train.append(split[0])
    X_test.append(split[1])
    y_train.append(split[2])
    y_test.append(split[3])


np.concat()

In [ ]:
# TODO: is normalization really useful?

X = np.concatenate(X, axis=0)
y = np.concatenate(y, axis=0)
y = y[:, 0]
print(X.shape)
print(y.shape)

In [11]:
from sklearn.model_selection import train_test_split


X_train, X_test_val, y_train, y_test_val = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

X_test, X_val, y_test, y_val = train_test_split(
    X_test_val, y_test_val, test_size=0.5, random_state=42, shuffle=True
)


In [4]:
mean = np.mean(X_train, axis=(0, 1))
std = np.std(X_train, axis=(0, 1))

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std


In [5]:
X_train.dtype

dtype('float64')

In [12]:
from sklearn.preprocessing import LabelEncoder

lb = LabelEncoder()
y_train_encoded = lb.fit_transform(y_train)
y_val_encoded = lb.transform(y_val)
y_test_encoded = lb.transform(y_test)

y_train_encoded

array([2, 4, 1, ..., 3, 1, 1], shape=(462563,))

In [8]:
min(y_train_encoded)

np.int64(0)

In [13]:
import torch
import torch.nn as nn


class CNN(nn.Module):
    def __init__(self, channel_in=12, kernel_size=7):
        super().__init__()
        # 64 * 12
        self.conv1 = nn.Conv1d(
            in_channels=channel_in,
            out_channels=128,
            kernel_size=kernel_size,
            stride=1,
            padding="same",
        )
        # => 64 * 64
        self.bn1 = nn.LazyBatchNorm1d()
        self.bn2 = nn.LazyBatchNorm1d()
        self.bn3 = nn.LazyBatchNorm1d()
        self.bn4 = nn.LazyBatchNorm1d()

        self.maxpool = nn.MaxPool1d(kernel_size=2)

        self.conv2 = nn.Conv1d(
            in_channels=128,
            out_channels=64,
            kernel_size=kernel_size,
            stride=1,
            padding="same",
        )

        self.conv3 = nn.Conv1d(
            in_channels=64,
            out_channels=32,
            kernel_size=kernel_size,
            stride=1,
            padding="same",
        )

        self.conv4 = nn.Conv1d(
            in_channels=32,
            out_channels=32,
            kernel_size=kernel_size,
            stride=1,
            padding="same",
        )
        # => 64 * 32

        self.fc1 = nn.LazyLinear(128)
        self.fc2 = nn.Linear(128, 5)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu(x)
        x = x.mean(dim=2)
        x = self.relu(self.fc1(x))
        return self.fc2(x)


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import Lambda, Compose

torch.manual_seed(42)

transform = Compose(
    [
        torch.FloatTensor,
        Lambda(lambda x: x.permute([1, 0])),
    ]
)


class DreamtDataset(Dataset):
    def __init__(
        self, X, y, mean=None, std=None, transform=None, target_transform=None
    ):
        super().__init__()
        self.X = X
        self.y = y
        self.transform = transform
        self.target_transform = target_transform

    def __getitem__(self, index):
        x = self.X[index]
        y = self.y[index]
        if self.transform:
            x = self.transform(self.X[index])

        if self.target_transform:
            y = self.target_transform(self.y[index])

        return x, y

    def __len__(self):
        return self.X.shape[0]


train_ds = DreamtDataset(X_train, y_train_encoded, transform=transform)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)

val_ds = DreamtDataset(X_val, y_val_encoded, transform=transform)
val_dl = DataLoader(val_ds, batch_size=1024)

In [15]:
sample, _ = next(iter(train_dl))

sample.dtype

torch.float32

In [3]:
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def train_model(model, train_dl, val_dl, epochs, lr=0.001, device="cpu"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(reduction="sum")
    for epoch in tqdm(range(epochs)):
        model.train()
        empirical_risk = 0.0
        for X, y in train_dl:
            X = X.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            empirical_risk += loss.item()

        empirical_risk /= len(train_dl.dataset)
        print("Train loss: %.3f" % (empirical_risk))
        if (epoch + 1) % 3 == 0:
            model.eval()
            val_empirical_loss = 0.0
            correct = 0
            with torch.no_grad():
                for X, y in val_dl:
                    X = X.to(DEVICE)
                    y = y.to(DEVICE)
                    pred = model(X)
                    loss = criterion(pred, y)
                    val_empirical_loss += loss.item()
                    correct += (torch.argmax(pred, dim=1) == y).sum().item()
                val_empirical_loss /= len(val_dl.dataset)
                val_acc = correct / len(val_dl.dataset)
                print("Val loss: %.3f, val_acc: %.3f" % (val_empirical_loss, val_acc))


model = CNN(channel_in=7)
model.to(DEVICE)

train_model(model, train_dl, val_dl, epochs=20, device=DEVICE)

NameError: name 'torch' is not defined

In [ ]:
from sklearn.metrics import f1_score


def test_model(model, test_dl, device="cpu"):
    criterion = nn.CrossEntropyLoss(reduction="sum")
    model.eval()
    generalization_error = 0.0
    correct = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X, y in test_dl:
            X = X.to(device)
            y = y.to(device)
            logits = model(X)
            loss = criterion(logits, y)
            pred = torch.argmax(logits, dim=1)
            correct += (pred == y).sum().item()
            generalization_error += loss.item()
            all_preds.append(pred.cpu())
            all_targets.append(y.cpu())

        y_pred = torch.cat(all_preds).numpy()
        y_true = torch.cat(all_targets).numpy()
        generalization_error /= len(test_dl.dataset)
        accuracy = correct / len(test_dl.dataset)
        print(
            "Generalization Error: %.3f, Accuracy %.3f"
            % (generalization_error, accuracy)
        )

    return y_true, y_pred


test_ds = DreamtDataset(X_test, y_test_encoded, transform=transform)
test_dl = DataLoader(test_ds, batch_size=1024)
y_true, y_pred = test_model(model, test_dl, DEVICE)


Generalization Error: 0.382, Accuracy 0.852


0.8451129772747628

In [26]:
f1_score(y_true, y_pred, average="macro")

0.7573013810346605